# Process

Process scans (HTR and OCR)

## 1. HTR with Claude

In [ ]:
import anthropic
from datetime import datetime
from dotenv import load_dotenv
import json
import os
from pathlib import Path
import regex
from collections import defaultdict

In [ ]:
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
def clear_claude_storage():
    try:
        files = client.beta.files.list()
        for file in files:
            client.beta.files.delete(file.id)
    except:
        pass

In [ ]:
def save_json(results_json, name="output", suffix="_" + datetime.strftime(datetime.now(), "%Y%m%d")):
    with open(f"{name}{suffix}.json", "w", encoding='utf-8') as f:
        json.dump(results_json, f, ensure_ascii=False, indent=2)

In [ ]:
def read_json(file_name):
    with open(file_name, "r", encoding='utf-8') as infile:
        return json.load(infile)

In [ ]:
def send_claude_prompt(prompt, upload_file_name):
    try:
        upload_response = client.beta.files.upload(file=Path(upload_file_name))
        message = client.beta.messages.create(
            model="claude-sonnet-5",
            max_tokens=1024,
            messages=[
                {"role": "user", 
                 "content": [
                    {"type": "image",
                     "source": {"type": "file",
                                "file_id": upload_response.id
                               }
                    },
                    {"type": "text", "text": prompt}
                  ]}],
            betas=["files-api-2025-04-14"]
        )
    finally:
        clear_claude_storage()
    text_blocks = [block.text for block in message.content if block.type == "text"]
    return "\n".join(text_blocks)

In [ ]:
def add_comment(person_dict, comment_prefix, comment_suffix):
    if comment_prefix:
        if comment_suffix:
            person_dict["comment"] = " ".join([comment_prefix, comment_suffix])
        else:
            person_dict["comment"] = comment_prefix
    elif comment_suffix:
        person_dict["comment"] = comment_suffix

In [ ]:
def add_page_number(person_dict, page_number, sample_file_name):
    sample_file_name_parts = regex.split(r"[_.]", sample_file_name)
    sample_file_name_parts[-2] = str(page_number).zfill(len(sample_file_name_parts[-2]))
    sample_file_name_parts[-2] += "." + sample_file_name_parts[-1]
    sample_file_name_parts.pop()
    person_dict["scan_file"] = "_".join(sample_file_name_parts)
    person_dict["page_number"] = page_number

In [ ]:
def str2dict(string, page_number, sample_file_name):
    groups = regex.search(r"^(.*)```json(.*)```(.*)$", string.strip(), flags=regex.DOTALL)
    person_dict = json.loads(groups.group(2))
    add_comment(person_dict, groups.group(1).strip(), groups.group(1).strip())
    add_page_number(person_dict, page_number, sample_file_name)
    return person_dict

In [ ]:
def claude2json(results):
    results_json = []
    for page_number, result in results.items():
        results_json.append(str2dict(result, page_number, "sample_file_name"))
    return results_json

In [ ]:
def read_key_value_lists(path):
    """
    Read a text file where each line has two or more space-separated
    tokens; only the first two matter. Returns a dict mapping the first
    token (key) to a list of second tokens seen for that key.
    Lines with fewer than 2 tokens are skipped.
    """
    result = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            tokens = line.split()
            if len(tokens) < 2:
                continue
            key, value = tokens[0], tokens[1]
            result[key].append(value)
    return dict(result)

In [ ]:
source_dir = "../memories_crawl/scans/bhic/Werkendam/deel_1924-1927/out_L/selected"
model = "claude-sonnet-4-6"
processed = read_key_value_lists("mistral_log.txt")

results = {}
for file_name in sorted(os.listdir(source_dir)):
    if file_name in processed and model in processed[file_name]:
        continue
    file_name_with_dir = os.path.join(source_dir, file_name)
    page_nbr = int(file_name[-14:-9])

    prompt = f"""Dear Claude, attached you will find a scan which contains the phrase 
'11. Opgave van den staat des boedels' in the top left. To the right of the phrase,
you will find some numbers denoting money values, one above each other. Usually there
are three numbers but sometimes also four, two or five. The numbers are preceded 
by an `f` for Dutch guilders or a double quote indicating a copy of the element above 
it (the `f`). We do not need these characters. Most numbers contain a decimal 
marker which could be a comma or a period. Some numbers have a hyphen behind the 
decimal marker, which stands for zero cents. Please replace the hyphen by "00" in
the output. A few numbers finish with a superscript `5` indicating half a cent. Please 
return a line with a name of the file, which is: {file_name}, the model name which is 
'{model}' and the numbers, separated by single spaces, and nothing else. If 
the second number is missing, fill in the value 0.00 for this number. When you see the 
handwritten word "Nadelig" in front of the third number, the third number is negative 
and a negative sign should be placed in front of it."""

    results[page_nbr] = send_claude_prompt(prompt, file_name_with_dir)
    if results[page_nbr]:
        print(results[page_nbr])
        with open("mistral_log.txt", "a")  as logfile:
            print(results[page_nbr], file=logfile)
    else:
        print(f"no result for {page_nbr}!")
        del results[page_nbr]
    
results_json = claude2json(results)
save_json(results_json)

## 2. Process with Mistral

In [ ]:
import os
import base64
import regex
from collections import defaultdict
from mistralai.client import Mistral
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# 1. Initialize the client
client = Mistral()

In [ ]:
# 3. Read and encode your image
model = "ministral-14b-latest" # "pixtral-12b-2409"
source_dir = "../memories_crawl/scans/bhic/Werkendam/deel_1924-1927/out_L/selected"
processed = read_key_value_lists("mistral_log.txt")

results = {}
for file_name in sorted(os.listdir(source_dir)):
    if file_name in processed and model in processed[file_name]:
        continue
    file_name_with_dir = os.path.join(source_dir, file_name)
    page_nbr = int(file_name[-14:-9])

    prompt = f"""Dear Mistral, attached you will find a scan which contains the phrase 
'11. Opgave van den staat des boedels' in the top left. To the right of the phrase,
you will find some numbers denoting money values, one above each other. Usually there
are three numbers but sometimes also four, two or five. The numbers are preceded 
by an `f` for Dutch guilders or a double quote indicating a copy of the element above 
it (the `f`). We do not need these characters. Most numbers contain a decimal 
marker which could be a comma or a period. Some numbers have a hyphen behind the 
decimal marker, which stands for zero cents. Please replace the hyphen by "00" in
the output. A few numbers finish with a superscript `5` indicating half a cent. Please 
return a line with a name of the file, which is: {file_name}, the model name which is 
'{model}' and the numbers, separated by single spaces, and nothing else. If 
the second number is missing, fill in the value 0.00 for this number. When you see the 
handwritten word "Nadelig" in front of the third number, the third number is negative 
and a negative sign should be placed in front of it."""

    
    with open(file_name_with_dir, "rb") as image_file:
        image_base64 = base64.b64encode(image_file.read()).decode("utf-8")
    # 4. Call the API with a vision-capable model
    response = client.chat.complete(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_base64}"}
                ]
            }
        ]
    )
    # 5. Print the result and save it to a log file
    print(response.choices[0].message.content)

    with open("mistral_log.txt", "a")  as logfile:
        print(regex.sub('\n', ' ', response.choices[0].message.content), file=logfile)

## 3. Process with Gemini

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv() 

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
model = "gemini-3.6-flash"
source_dir = "../memories_crawl/scans/bhic/Werkendam/deel_1924-1927/out_L/selected"
processed = read_key_value_lists("mistral_log.txt")

results = {}
for file_name in sorted(os.listdir(source_dir)):
    if file_name in processed and model in processed[file_name]:
        continue
    file_name_with_dir = os.path.join(source_dir, file_name)
    page_nbr = int(file_name[-14:-9])
    image = types.Part.from_bytes(
        data=open(file_name_with_dir, "rb").read(),
        mime_type="image/jpeg",
    )
    prompt = f"""Dear Gemini, attached you will find a scan which contains the phrase 
'11. Opgave van den staat des boedels' in the top left. To the right of the phrase,
you will find some numbers denoting money values, one above each other. Usually there
are three numbers but sometimes also four, two or five. The numbers are preceded 
by an `f` for Dutch guilders or a double quote indicating a copy of the element above 
it (the `f`). We do not need these characters. Most numbers contain a decimal 
marker which could be a comma or a period. Some numbers have a hyphen behind the 
decimal marker, which stands for zero cents. Please replace the hyphen by "00" in
the output. A few numbers finish with a superscript `5` indicating half a cent. Please 
return a line with a name of the file, which is: {file_name}, the model name which is 
'{model}' and the numbers, separated by single spaces, and nothing else. If 
the second number is missing, fill in the value 0.00 for this number. When you see the 
handwritten word "Nadelig" in front of the third number, the third number is negative 
and a negative sign should be placed in front of it."""

    response = client.models.generate_content(
        model=model,
        contents=[image, prompt]
    )
    if not response.text:
        print(f"No result for page {page_nbr}")
    else:
        print(response.text)
        with open("mistral_log.txt", "a")  as logfile:
            print(regex.sub('\n', ' ', response.text), file=logfile)

Costs run 1 (227 images): 1.67 (18.33): 0.007

## 4. Compare with gold data

In [ ]:
import sys
from collections import defaultdict
from typing import Dict, List, Tuple

from parse_log import parse_log


def max_bipartite_matches(gold: List[float], pred: List[float], tolerance: float = 1e-6) -> int:
    """
    Return the size of the maximum matching between `gold` and `pred`,
    where an edge exists between gold[i] and pred[j] iff they are equal
    within `tolerance`. Each element on either side is used at most once.
    Uses Kuhn's algorithm (fine for the small per-line lists here).
    """
    n, m = len(gold), len(pred)
    adjacency = [
        [j for j in range(m) if abs(gold[i] - pred[j]) <= tolerance]
        for i in range(n)
    ]
    match_of_pred = [-1] * m  # match_of_pred[j] = index i of gold matched to pred j

    def try_augment(i: int, visited: List[bool]) -> bool:
        for j in adjacency[i]:
            if not visited[j]:
                visited[j] = True
                if match_of_pred[j] == -1 or try_augment(match_of_pred[j], visited):
                    match_of_pred[j] = i
                    return True
        return False

    matched = 0
    for i in range(n):
        visited = [False] * m
        if try_augment(i, visited):
            matched += 1
    return matched


def load_gold(path: str) -> Dict[str, List[float]]:
    """filename -> list of gold numbers (model column is expected to be 'gold')."""
    gold: Dict[str, List[float]] = {}
    for filename, model, numbers in parse_log(path):
        if filename in gold:
            print(f"Warning: duplicate gold entry for {filename!r}, keeping the first.", file=sys.stderr)
            continue
        gold[filename] = numbers
    return gold


def satisfies_checksum(numbers: List[float], tolerance: float = 1e-6) -> bool:
    """
    True iff numbers[0] == numbers[1] + numbers[2] (within tolerance).
    A model that recognized fewer than 3 numbers is treated as violating
    the constraint (per your instruction), since there's nothing to check.
    """
    if len(numbers) < 3:
        return False
    return abs(numbers[0] - (numbers[1] + numbers[2])) <= tolerance


def compare(log_path: str, gold_path: str, tolerance: float = 1e-6, checksum_tolerance: float = 1e-6):
    gold = load_gold(gold_path)
    gold_filenames = set(gold.keys())

    # per_model[model] = list of (filename, gold_numbers, pred_numbers, matched, checksum_ok)
    per_model: Dict[str, List[Tuple[str, List[float], List[float], int, bool]]] = defaultdict(list)
    # model_seen[model] = every filename that model has ANY entry for in
    # log_path, whether or not it had a matching gold entry -- this is
    # what lets us tell "never attempted" apart from "attempted, no gold".
    model_seen: Dict[str, set] = defaultdict(set)

    for filename, model, pred_numbers in parse_log(log_path):
        model_seen[model].add(filename)
        gold_numbers = gold.get(filename)
        if gold_numbers is None:
            print(f"Warning: no gold entry for {filename!r} (model {model!r}); skipping.", file=sys.stderr)
            continue
        matched = max_bipartite_matches(gold_numbers, pred_numbers, tolerance)
        checksum_ok = satisfies_checksum(pred_numbers, checksum_tolerance)
        per_model[model].append((filename, gold_numbers, pred_numbers, matched, checksum_ok))

    # missing[model] = gold filenames that model never produced any line for
    # at all -- the case the old script silently didn't report.
    missing: Dict[str, List[str]] = {
        model: sorted(gold_filenames - seen) for model, seen in model_seen.items()
    }

    return per_model, missing


def print_report(per_model, missing, show_per_file: bool = False):
    print(f"{'model':<21} {'correct':>8} {'gold total':>11} {'pred total':>11} "
          f"{'recall':>8} {'precision':>10} {'checksum OK':>13} {'failed':>9}")

    # Union of keys: a model could in principle have entries in one dict but
    # not the other (e.g. a model that appears in the log but matched no
    # gold filenames at all would have no per_model rows, yet still needs
    # its missing count reported).
    all_models = sorted(set(per_model) | set(missing))

    for model in all_models:
        rows = per_model.get(model, [])
        total_correct = sum(r[3] for r in rows)
        total_gold = sum(len(r[1]) for r in rows)
        total_pred = sum(len(r[2]) for r in rows)
        total_files = len(rows)
        checksum_ok_count = sum(1 for r in rows if r[4])
        recall = total_correct / total_gold if total_gold else float('nan')
        precision = total_correct / total_pred if total_pred else float('nan')
        checksum_pct = checksum_ok_count / total_files if total_files else float('nan')
        missing_count = len(missing.get(model, []))
        print(f"{model:<21} {total_correct:>8} {total_gold:>11} {total_pred:>11} "
              f"{recall:>8.1%} {precision:>10.1%} "
              f"{checksum_ok_count:>5}/{total_files:<3} ({checksum_pct:.1%}) {missing_count:>5}")

    if show_per_file:
        for model in all_models:
            rows = per_model.get(model, [])
            print(f"\n--- {model} ---")
            for filename, gold_numbers, pred_numbers, matched, checksum_ok in rows:
                checksum_label = "checksum OK" if checksum_ok else "checksum FAIL"
                if not checksum_ok:
                    print(f"{matched}/{len(gold_numbers)} correct  {checksum_label:<14} "
                          f"gold={gold_numbers}  pred={pred_numbers}  {filename}")
            if missing.get(model):
                print(f"  missing entirely ({len(missing[model])}):")
                for filename in missing[model]:
                    print(f"    {filename}")


if True:
    log_path = "mistral_log.txt"
    gold_path = "mistral_correct.txt"
    per_model, missing = compare(log_path, gold_path)
    print_report(per_model, missing, show_per_file=False)

Notes:

* there are 6 gold pages for which the checksum is incorrect: 240, 531, 1068 and 1085 (missing second number), 436 (rounding error) and 1237 (completely off)
* this book contains 552 deeds and 224/552 is only 40% for which we have this type of form page
* missed pages: 478 (5), 1158 (4-6) and 1237 (both)
* pages with two numbers: 240, 473, 513, 531, 591, 618, 986, 1068, 1085,1190 (10)
* pages with Nadelig: 240, 638, 786, 949, 1120, 1154 (6)